# GeoCrop crop-type model: accuracy boost (one run, one result table)

**What this notebook does.** It takes the same Sentinel-2 monthly file and the same CDL labels as the benchmark, uses the **same spatial block split** (so test pixels are never next to training pixels) and tries four improvements, one after the other:

1. **More inputs per month:** four extra spectral indices (NDMI, NDRE, EVI, NDWI) and a flag telling the model which months were really observed (not filled in).
2. **Neighbourhood inputs:** the mean and spread of NDVI, B8 and B11 over the surrounding 3x3 and 7x7 pixels (fields are uniform, a single pixel is noisy).
3. **Cleaner training labels:** field-edge pixels (mixed pixels) are removed from the *training* data only. Validation and test keep every pixel.
4. **Several models averaged** (LightGBM + Random Forest + MLP), and an optional **neighbour smoothing** of the final map (reported separately).

**How choices are made (no test-set tuning).** An improvement is kept only if it raises the macro-F1 on the *validation* blocks. The test blocks are scored and printed for every step so you can see everything, but they never decide anything.

**To run:** Save Version -> Save & Run All. Nothing to change. For area 2 change `AOI = 1` to `AOI = 2` in the next cell.
At the very bottom, copy everything below the line `===== RESULTS =====` and send it.

In [11]:
import os, sys, json, time, re, gc, random, warnings, subprocess, platform
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import psutil
from IPython.display import display
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
T0 = time.time()
def log(msg): print(f"[{(time.time() - T0) / 60:5.1f} min] {msg}", flush=True)

try:
    import rasterio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=False)
    import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import uniform_filter, maximum_filter, minimum_filter, binary_dilation
import torch
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

AOI = 2                                   # <- the only thing you may change: 1 or 2

CFG = dict(
    AOI=AOI, INPUT_ROOT="/kaggle/input", OUTPUT_DIR=f"/kaggle/working/boost_outputs_aoi{AOI}",
    S2_GLOB=f"**/aoi{AOI}_monthly_*.tif", CDL_GLOB=f"**/usa_cdl_aoi{AOI}*.tif",
    SEED=42,
    # same data rules as the benchmark notebook
    CROP_ONLY=True, MIN_PIXELS_PER_CLASS=5000, USE_OTHER_CLASS=True, TS_MAX_MISSING_FRAC=0.5,
    BLOCK_SIZE_PX=100, SPLIT_FRACS=(0.70, 0.15, 0.15), BUFFER_PX=3, MAX_TRAIN_PER_CLASS=30000,
    # models
    GBM_ROUNDS=400, RF_TREES=100, MLP_EPOCHS=30, MLP_PATIENCE=5, BATCH_SIZE=1024,
    # the new parts
    NBR_WINDOWS=(3, 7),                   # neighbourhood sizes in pixels
    EDGE_PX=1,                            # field-edge pixels (1 = the one-pixel border of every field) are dropped from TRAINING
    MIN_GAIN=0.002,                       # an improvement is kept only if validation macro-F1 rises by at least this much
    SMOOTH_WINDOWS=(1, 3, 5, 7, 9, 11),   # neighbour-smoothing sizes tried on the validation blocks
    FAST_DEV=False,                       # True = tiny smoke test, not for real results
)
CFG.update(json.loads(os.environ.get("BOOST_CFG_JSON", "{}")))
if CFG["FAST_DEV"]:
    CFG.update(MAX_TRAIN_PER_CLASS=2500, GBM_ROUNDS=60, RF_TREES=30, MLP_EPOCHS=6, MLP_PATIENCE=2)
OUT = Path(CFG["OUTPUT_DIR"])
for d in ["tables", "predictions"]: (OUT / d).mkdir(parents=True, exist_ok=True)
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"area {AOI} | device {DEVICE} | cpus {os.cpu_count()} | ram {psutil.virtual_memory().total / 1e9:.0f} GB")

[  0.0 min] area 2 | device cuda | cpus 4 | ram 34 GB


## 1. Load the Sentinel-2 monthly file and the CDL labels

In [12]:
def show_tree(root, max_depth=3, max_items=60):
    root = Path(root); n = 0
    if not root.exists(): print(f"(missing) {root}"); return
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth: continue
        print("  " * (depth - 1) + p.name + (f"  [{p.stat().st_size / 1e6:,.1f} MB]" if p.is_file() else "/")); n += 1
        if n >= max_items: print("  ..."); break

def find_file(pattern, root, what):
    hits = sorted(Path(root).glob(pattern))
    if not hits:
        print(f"!! no {what} found with pattern '{pattern}' under {root}. What is there:"); show_tree(root)
        raise FileNotFoundError(what)
    if len(hits) > 1: print(f"note: several candidates {[h.name for h in hits]}; using {hits[0].name}")
    return hits[0]

S2_PATH = find_file(CFG["S2_GLOB"], CFG["INPUT_ROOT"], "satellite raster")
CDL_PATH = find_file(CFG["CDL_GLOB"], CFG["INPUT_ROOT"], "label raster (CDL)")
print("satellite:", S2_PATH, "\nlabels   :", CDL_PATH)

with rasterio.open(S2_PATH) as ds:
    DESC = list(ds.descriptions)
    ARR = ds.read().astype(np.float32)
    S2_TRANSFORM, S2_CRS = ds.transform, ds.crs
    if ds.nodata is not None: ARR[ARR == ds.nodata] = np.nan
    ARR[ARR == -32768] = np.nan                     # int16 exports use -32768 for "no clear observation"

TS_PAT = re.compile(r"^(\d{4})-(\d{2})_(.+)$")
parsed = [TS_PAT.match(d) if d else None for d in DESC]
assert ARR.shape[0] > 0 and all(parsed), "band names must look like YYYY-MM_BAND (the monthly export). Got: %s" % DESC[:6]
MONTHS = sorted({(int(m.group(1)), int(m.group(2))) for m in parsed})
BAND_NAMES = []
for m in parsed:
    if m.group(3) not in BAND_NAMES: BAND_NAMES.append(m.group(3))
T, C = len(MONTHS), len(BAND_NAMES)
index = {(MONTHS.index((int(m.group(1)), int(m.group(2)))), BAND_NAMES.index(m.group(3))): i for i, m in enumerate(parsed)}
assert len(index) == T * C, "irregular band layout"
H, W = ARR.shape[1:]
print(f"grid {H} x {W} | months T={T} | bands per month C={C}: {BAND_NAMES}")
sel = np.array([[index[(t, c)] for c in range(C)] for t in range(T)]).reshape(-1)
CUBE = np.ascontiguousarray(ARR[sel].reshape(T, C, H, W).transpose(2, 3, 0, 1)); del ARR; gc.collect()

for c, b in enumerate(BAND_NAMES):                  # units: reflectance 0-1, indices in their natural range
    v = CUBE[::6, ::6, :, c]; v = v[np.isfinite(v)]
    if v.size == 0: continue
    s = (1e4 if float(np.percentile(v, 99)) > 2.5 else 1.0) if re.fullmatch(r"B\d+A?", b) else (1e4 if float(np.abs(v).max()) > 5 else 1.0)
    if s != 1.0: CUBE[..., c] /= s
fin = np.isfinite(CUBE)
print(f"missing cells: {100 * (1 - fin.mean()):.1f}%")
PIX_FRAC_OK = fin.all(-1).mean(-1); del fin; gc.collect()
BI = {b: i for i, b in enumerate(BAND_NAMES)}
for need in ["B2", "B3", "B4", "B5", "B8", "B11", "NDVI"]:
    assert need in BI, f"band {need} is missing in the file; found {BAND_NAMES}"

with rasterio.open(CDL_PATH) as ds:
    cdl_src, cdl_tf, cdl_crs = ds.read(1), ds.transform, ds.crs
if cdl_crs == S2_CRS and cdl_src.shape == (H, W) and cdl_tf.almost_equals(S2_TRANSFORM):
    CDL_IMG = cdl_src.astype(np.uint8); print("label grid = image grid")
else:
    print("-> labels are re-gridded onto the image grid with nearest neighbour")
    CDL_IMG = np.zeros((H, W), np.uint8)
    reproject(source=cdl_src, destination=CDL_IMG, src_transform=cdl_tf, src_crs=cdl_crs, dst_transform=S2_TRANSFORM, dst_crs=S2_CRS,
              resampling=Resampling.nearest, src_nodata=0, dst_nodata=0)
del cdl_src; gc.collect()
log("data loaded")

satellite: /kaggle/input/datasets/tahamalik123/geocrop/aoi2_monthly_2022.tif 
labels   : /kaggle/input/datasets/tahamalik123/geocrop/usa_cdl_aoi2.tif
grid 955 x 1269 | months T=12 | bands per month C=11: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI']
missing cells: 0.2%
label grid = image grid
[  0.2 min] data loaded


## 2. Cleaning and classes (identical rules to the benchmark)

In [13]:
CDL_NAMES = {1:"Corn",2:"Cotton",3:"Rice",4:"Sorghum",5:"Soybeans",6:"Sunflower",10:"Peanuts",11:"Tobacco",12:"Sweet Corn",13:"Pop/Orn Corn",14:"Mint",
 21:"Barley",22:"Durum Wheat",23:"Spring Wheat",24:"Winter Wheat",25:"Other Small Grains",26:"WinWheat/Soybeans",27:"Rye",28:"Oats",29:"Millet",31:"Canola",
 32:"Flaxseed",33:"Safflower",34:"Rape Seed",35:"Mustard",36:"Alfalfa",37:"Other Hay",38:"Camelina",39:"Buckwheat",41:"Sugarbeets",42:"Dry Beans",43:"Potatoes",
 44:"Other Crops",45:"Sugarcane",46:"Sweet Potatoes",47:"Misc Vegs & Fruits",48:"Watermelons",49:"Onions",50:"Cucumbers",51:"Chick Peas",52:"Lentils",53:"Peas",
 54:"Tomatoes",55:"Caneberries",56:"Hops",57:"Herbs",58:"Clover/Wildflowers",59:"Sod/Grass Seed",60:"Switchgrass",61:"Fallow/Idle Cropland",66:"Cherries",
 67:"Peaches",68:"Apples",69:"Grapes",70:"Christmas Trees",71:"Other Tree Crops",72:"Citrus",74:"Pecans",75:"Almonds",76:"Walnuts",77:"Pears",
 111:"Open Water",112:"Perennial Ice/Snow",121:"Developed/Open Space",122:"Developed/Low",123:"Developed/Med",124:"Developed/High",131:"Barren",
 141:"Deciduous Forest",142:"Evergreen Forest",143:"Mixed Forest",152:"Shrubland",176:"Grassland/Pasture",190:"Woody Wetlands",195:"Herbaceous Wetlands",
 204:"Pistachios",205:"Triticale",206:"Carrots",207:"Asparagus",208:"Garlic",209:"Cantaloupes",210:"Prunes",211:"Olives",212:"Oranges",213:"Honeydew",
 214:"Broccoli",215:"Avocados",216:"Peppers",217:"Pomegranates",218:"Nectarines",219:"Greens",220:"Plums",221:"Strawberries",222:"Squash",223:"Apricots",
 224:"Vetch",225:"WinWheat/Corn",226:"Oats/Corn",227:"Lettuce",229:"Pumpkins",241:"Corn/Soybeans",242:"Blueberries",243:"Cabbage",244:"Cauliflower",
 245:"Celery",246:"Radishes",247:"Turnips",248:"Eggplants",249:"Gourds",250:"Cranberries"}
NON_CROP_CODES = {111, 112, 121, 122, 123, 124, 131, 141, 142, 143, 152, 176, 190, 195}
def cdl_name(code): return CDL_NAMES.get(int(code), f"CDL_{int(code)}")

S2_IDX = [c for c, b in enumerate(BAND_NAMES) if re.fullmatch(r"B\d+A?", b)]
sat_ok = PIX_FRAC_OK >= (1.0 - CFG["TS_MAX_MISSING_FRAC"])
with np.errstate(invalid="ignore"):
    rng_bad = ((CUBE[..., S2_IDX] < -0.05) | (CUBE[..., S2_IDX] > 2.0)).any(axis=(-1, -2))
alive = sat_ok & ~rng_bad & (CDL_IMG > 0)
if CFG["CROP_ONLY"]: alive &= ~np.isin(CDL_IMG, list(NON_CROP_CODES))
print(f"labelled, usable pixels: {int(alive.sum()):,} of {H * W:,}")
cnt = pd.Series(CDL_IMG[alive]).value_counts()
keep_codes = [int(c) for c in cnt[cnt >= CFG["MIN_PIXELS_PER_CLASS"]].index]
rare_codes = [int(c) for c in cnt[cnt < CFG["MIN_PIXELS_PER_CLASS"]].index]
code_to_idx = {c: i for i, c in enumerate(keep_codes)}
CLASS_CODES, CLASS_NAMES = list(keep_codes), [cdl_name(c) for c in keep_codes]
Y_IMG = np.full((H, W), -1, np.int16)
mapped = np.array([code_to_idx.get(c, -1) for c in range(256)])[CDL_IMG[alive].astype(int)]
if CFG["USE_OTHER_CLASS"] and rare_codes and cnt[rare_codes].sum() > 0:
    mapped = np.where(mapped < 0, len(keep_codes), mapped); CLASS_CODES.append(-1); CLASS_NAMES.append("Other (rare classes)")
Y_IMG[alive] = mapped; Y_IMG[Y_IMG < 0] = -1
K = len(CLASS_NAMES)
cls_tbl = pd.DataFrame({"class_idx": range(K), "cdl_code": CLASS_CODES, "name": CLASS_NAMES, "pixels": [int((Y_IMG == k).sum()) for k in range(K)]})
cls_tbl["share_%"] = 100 * cls_tbl.pixels / cls_tbl.pixels.sum()
display(cls_tbl); cls_tbl.to_csv(OUT / "tables" / "classes.csv", index=False)

# field edges (mixed pixels): a pixel whose neighbourhood contains a different CDL code
_size = 2 * CFG["EDGE_PX"] + 1
EDGE_IMG = maximum_filter(CDL_IMG, size=_size) != minimum_filter(CDL_IMG, size=_size)
ROWS, COLS = np.nonzero(Y_IMG >= 0)
Y_ALL = Y_IMG[ROWS, COLS].astype(np.int64)
EDGE_AT = EDGE_IMG[ROWS, COLS]
print(f"share of labelled pixels on a field edge: {100 * EDGE_AT.mean():.1f}%")

labelled, usable pixels: 948,425 of 1,211,895


,class_idx,cdl_code,name,pixels,share_%
0,0,31,Canola,341759,36.0344
1,1,23,Spring Wheat,325662,34.3371
2,2,5,Soybeans,135635,14.3011
3,3,61,Fallow/Idle Cropland,88042,9.2830
4,4,21,Barley,20116,2.1210
5,5,53,Peas,12191,1.2854
6,6,42,Dry Beans,9831,1.0366
7,7,6,Sunflower,6679,0.7042
8,8,-1,Other (rare classes),8510,0.8973


share of labelled pixels on a field edge: 31.0%


## 3. Build the extra inputs (from the images only, no labels are used here)

In [14]:
EPS = 1e-6
def nd(a, b): return (a - b) / (a + b + EPS)
def vol(name): return CUBE[..., BI[name]]                         # (H, W, T) view

def monthly_filter(v, size, kind):
    """Mean or standard deviation over a size x size window, month by month, ignoring missing pixels."""
    out = np.empty(v.shape, np.float32)
    for t in range(v.shape[2]):
        a = np.ascontiguousarray(v[:, :, t]); m = np.isfinite(a)
        x = np.where(m, a, 0).astype(np.float32); w = m.astype(np.float32)
        den = uniform_filter(w, size=size, mode="nearest")
        mean = uniform_filter(x, size=size, mode="nearest") / np.maximum(den, 1e-6)
        if kind == "mean": res = mean
        else: res = np.sqrt(np.maximum(uniform_filter(x * x, size=size, mode="nearest") / np.maximum(den, 1e-6) - mean * mean, 0))
        out[:, :, t] = np.where(den > 1e-6, res, np.nan)
    return out

EXTRA_NAMES, EXTRA_GROUP, EXTRA_ARR = [], [], []
def add_extra(name, group, arr_hwt):
    EXTRA_NAMES.append(name); EXTRA_GROUP.append(group); EXTRA_ARR.append(arr_hwt[ROWS, COLS].astype(np.float32))   # (N, T)

with np.errstate(invalid="ignore", divide="ignore"):
    add_extra("NDMI", "idx", nd(vol("B8"), vol("B11")))
    add_extra("NDRE", "idx", nd(vol("B8"), vol("B5")))
    add_extra("EVI", "idx", np.clip(2.5 * (vol("B8") - vol("B4")) / (vol("B8") + 6 * vol("B4") - 7.5 * vol("B2") + 1.0), -1, 3))
    add_extra("NDWI", "idx", nd(vol("B3"), vol("B8")))
add_extra("OBS", "idx", np.isfinite(vol("B4")).astype(np.float32))       # 1 = this month was really observed, 0 = it will be filled in
for w in CFG["NBR_WINDOWS"]:
    for b in ["NDVI", "B8", "B11"]:
        add_extra(f"{b}_m{w}", "nbr", monthly_filter(vol(b), w, "mean"))
    add_extra(f"NDVI_s{w}", "nbr", monthly_filter(vol("NDVI"), w, "std"))
log(f"extra inputs built: {len(EXTRA_NAMES)} per month -> {EXTRA_NAMES}")

def fill_gaps_inplace(X, chunk=50_000):
    """Missing months: linear interpolation along time; gaps at the start/end copy the nearest observation. Works in place, chunk by chunk."""
    for s in range(0, len(X), chunk):
        x = X[s:s + chunk]; nan = np.isnan(x)
        if not nan.any(): continue
        n, Tn, Cn = x.shape; idx = np.arange(Tn)[None, :, None]
        prev = np.maximum.accumulate(np.where(~nan, idx, -1), axis=1)
        nxt = np.minimum.accumulate(np.where(~nan, idx, Tn)[:, ::-1], axis=1)[:, ::-1]
        pv = np.take_along_axis(x, np.clip(prev, 0, Tn - 1), 1); nv = np.take_along_axis(x, np.clip(nxt, 0, Tn - 1), 1)
        w = np.where(nxt > prev, (idx - prev) / np.maximum(nxt - prev, 1), 0.0)
        filled = np.where((prev >= 0) & (nxt < Tn), pv + (nv - pv) * w, np.where(prev >= 0, pv, nv))
        X[s:s + chunk] = np.where(nan, filled, x)

N_PIX, E = len(ROWS), len(EXTRA_ARR)
X_FULL = np.empty((N_PIX, T, C + E), np.float32)                  # (N, T, channels)
X_FULL[:, :, :C] = CUBE[ROWS, COLS]
for i, a in enumerate(EXTRA_ARR): X_FULL[:, :, C + i] = a
del EXTRA_ARR, CUBE; gc.collect()                                   # free memory early
fill_gaps_inplace(X_FULL)
assert np.isfinite(X_FULL).all(), "non-finite values remain after gap filling"
ALL_CH = BAND_NAMES + EXTRA_NAMES
CH_BASE = list(range(C))
CH_IDX = [C + i for i, g in enumerate(EXTRA_GROUP) if g == "idx"]
CH_NBR = [C + i for i, g in enumerate(EXTRA_GROUP) if g == "nbr"]
print(f"samples {len(X_FULL):,} | arrays (N, T, channels) = {X_FULL.shape} | base {len(CH_BASE)}, indices+flag {len(CH_IDX)}, neighbourhood {len(CH_NBR)} channels")

[  0.4 min] extra inputs built: 13 per month -> ['NDMI', 'NDRE', 'EVI', 'NDWI', 'OBS', 'NDVI_m3', 'B8_m3', 'B11_m3', 'NDVI_s3', 'NDVI_m7', 'B8_m7', 'B11_m7', 'NDVI_s7']
samples 948,425 | arrays (N, T, channels) = (948425, 12, 24) | base 11, indices+flag 5, neighbourhood 8 channels


## 4. Spatial block split (same rules and same seed as the benchmark)

In [15]:
def spatial_block_split(y_img, K, block, fracs, seed, buffer_px):
    valid = y_img >= 0
    r = np.arange(y_img.shape[0]) // block; c = np.arange(y_img.shape[1]) // block
    bid = (r[:, None] * (c.max() + 1) + c[None, :]).astype(np.int32); nb = int(bid.max()) + 1
    counts = np.zeros((nb, K)); np.add.at(counts, (bid[valid], y_img[valid].astype(int)), 1)
    totals = counts.sum(0); target = np.array(fracs)[:, None] * totals[None, :]; cur = np.zeros_like(target)
    rng = np.random.default_rng(seed); order = rng.permutation(nb)
    order = order[np.argsort(-counts.sum(1)[order], kind="stable")]
    assign = np.full(nb, -1)
    for b in order:
        if counts[b].sum() == 0: continue
        s = int(np.argmax((((target - cur) / np.maximum(target, 1)) * counts[b][None, :]).sum(1)))
        assign[b] = s; cur[s] += counts[b]
    split = np.full(y_img.shape, -1, np.int8); split[valid] = assign[bid[valid]]
    if buffer_px > 0:
        st = np.ones((3, 3), bool); is_te, is_va = split == 2, split == 1
        near_te = binary_dilation(is_te, st, iterations=buffer_px); near_va = binary_dilation(is_va, st, iterations=buffer_px)
        split[(split == 1) & near_te] = -2
        split[(split == 0) & (near_te | near_va)] = -2
    return split, assign

SPLIT_IMG, _ = spatial_block_split(Y_IMG, K, CFG["BLOCK_SIZE_PX"], CFG["SPLIT_FRACS"], CFG["SEED"], CFG["BUFFER_PX"])
SPLIT = SPLIT_IMG[ROWS, COLS]
IDX = {"train": np.where(SPLIT == 0)[0], "val": np.where(SPLIT == 1)[0], "test": np.where(SPLIT == 2)[0]}
print({s: f"{len(v):,}" for s, v in IDX.items()}, "| dropped by buffer:", int((SPLIT == -2).sum()))
tab = pd.DataFrame({"class": CLASS_NAMES, **{s: [int((Y_ALL[IDX[s]] == k).sum()) for k in range(K)] for s in IDX}}); display(tab)

def cap_per_class(idx, y, cap, seed):
    rng = np.random.default_rng(seed); keep = []
    for k in range(K):
        ii = idx[y[idx] == k]
        keep.append(rng.choice(ii, size=min(cap, len(ii)), replace=False) if len(ii) else ii)
    return np.sort(np.concatenate(keep))

{'train': '639,357', 'val': '138,979', 'test': '144,922'} | dropped by buffer: 25167


,class,train,val,test
0,Canola,230038,49210,52527
1,Spring Wheat,221402,48350,47912
2,Soybeans,91409,20231,20268
3,Fallow/Idle Cropland,59588,12159,14051
4,Barley,12674,3691,3255
5,Peas,8119,1591,2255
6,Dry Beans,5515,1518,2584
7,Sunflower,4543,1181,811
8,Other (rare classes),6069,1048,1259


## 5. Metrics and model helpers

In [16]:
def metrics(y, p):
    pr, rc, f1, s = precision_recall_fscore_support(y, p, labels=np.arange(K), zero_division=0); pres = s > 0
    return dict(acc=accuracy_score(y, p), f1=float(f1[pres].mean()), wf1=float((f1 * s).sum() / s.sum()))
def per_class(y, p):
    pr, rc, f1, s = precision_recall_fscore_support(y, p, labels=np.arange(K), zero_division=0)
    return pd.DataFrame(dict(cls=CLASS_NAMES, precision=pr, recall=rc, f1=f1, support=s))
def flat(X): return X.reshape(len(X), -1)
def full_proba(classes, proba):
    out = np.zeros((len(proba), K), np.float32); out[:, np.asarray(classes, dtype=int)] = proba; return out

def make_sets(ch, clean):
    """Arrays for one configuration. `clean` removes field-edge pixels from the TRAINING pool only."""
    pool = IDX["train"][~EDGE_AT[IDX["train"]]] if clean else IDX["train"]
    tr = cap_per_class(pool, Y_ALL, CFG["MAX_TRAIN_PER_CLASS"], CFG["SEED"])
    Xtr = X_FULL[tr][:, :, ch]
    mean = Xtr.reshape(-1, len(ch)).mean(0); std = np.maximum(Xtr.reshape(-1, len(ch)).std(0), 1e-6)
    f = lambda X: ((X - mean) / std).astype(np.float32)
    return dict(Xtr=f(Xtr), ytr=Y_ALL[tr], Xva=f(X_FULL[IDX["val"]][:, :, ch]), yva=Y_ALL[IDX["val"]],
                Xte=f(X_FULL[IDX["test"]][:, :, ch]), yte=Y_ALL[IDX["test"]], ch=list(ch), clean=clean)

def fit_lgbm(S, seed):
    est = lgb.LGBMClassifier(n_estimators=CFG["GBM_ROUNDS"], num_leaves=31, learning_rate=0.1, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, class_weight="balanced", random_state=seed, n_jobs=-1, verbose=-1)
    est.fit(flat(S["Xtr"]), S["ytr"], eval_set=[(flat(S["Xva"]), S["yva"])], callbacks=[lgb.early_stopping(30, verbose=False)])
    return est

def probs_of(est, S): return {k: full_proba(est.classes_, est.predict_proba(flat(S[x]))) for k, x in (("val", "Xva"), ("test", "Xte"))}

class MLPNet(nn.Module):
    def __init__(self, T, C, K, h=256, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(T * C, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(h, h // 2), nn.BatchNorm1d(h // 2), nn.ReLU(), nn.Dropout(drop), nn.Linear(h // 2, K))
    def forward(self, x): return self.net(x)

def mlp_probs(model, X, bs=8192):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.softmax(model(torch.from_numpy(np.ascontiguousarray(X[i:i + bs])).to(DEVICE)), 1).cpu())
    return torch.cat(out).numpy()

def fit_mlp(S, seed):
    set_seed(seed); Tn, Cn = S["Xtr"].shape[1:]
    model = MLPNet(Tn, Cn, K).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    cw = (len(S["ytr"]) / (K * np.maximum(np.bincount(S["ytr"], minlength=K), 1))).astype(np.float32)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, device=DEVICE))
    Xt, yt = torch.from_numpy(S["Xtr"]), torch.from_numpy(S["ytr"]); n = len(Xt); bs = CFG["BATCH_SIZE"]
    sub = np.sort(np.random.default_rng(seed).choice(len(S["yva"]), size=min(60000, len(S["yva"])), replace=False))
    gen = torch.Generator().manual_seed(seed); best, best_state, bad = -1.0, None, 0
    for ep in range(CFG["MLP_EPOCHS"]):
        model.train(); perm = torch.randperm(n, generator=gen)
        for i in range(0, n, bs):
            ii = perm[i:i + bs]
            if len(ii) < 2: continue
            opt.zero_grad(set_to_none=True); loss = lossf(model(Xt[ii].to(DEVICE)), yt[ii].to(DEVICE)); loss.backward(); opt.step()
        f1 = metrics(S["yva"][sub], mlp_probs(model, S["Xva"][sub]).argmax(1))["f1"]
        if f1 > best + 1e-4: best, bad, best_state = f1, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["MLP_PATIENCE"]: break
    model.load_state_dict(best_state); return model

## 6. The ladder: add one improvement at a time (LightGBM probe)
Step 0 is the benchmark setup (LightGBM on the 132 raw inputs), so you can check it against your earlier result. Each next step is kept only if **validation** macro-F1 rises by at least `MIN_GAIN`. Test numbers are shown for every step but are never used to decide.

In [17]:
LADDER, PROBS = [], {}
def run_step(name, ch, clean):
    t = time.time(); S = make_sets(ch, clean); est = fit_lgbm(S, CFG["SEED"]); pr = probs_of(est, S)
    mv, mt = metrics(S["yva"], pr["val"].argmax(1)), metrics(S["yte"], pr["test"].argmax(1))
    row = dict(step=name, inputs_per_month=len(ch), clean_train_labels=clean, rounds=int(est.booster_.current_iteration()),
               val_macro_f1=mv["f1"], test_acc=mt["acc"], test_macro_f1=mt["f1"], minutes=(time.time() - t) / 60)
    log(f"{name}: val F1 {mv['f1']:.4f} | test acc {mt['acc']:.4f} | test macro-F1 {mt['f1']:.4f}")
    return row, S, est, pr

row0, S_best, est_best, pr_best = run_step("0  benchmark setup (raw inputs)", CH_BASE, False)
row0["decision"] = "start"; LADDER.append(row0)
cur = dict(ch=list(CH_BASE), clean=False, val=row0["val_macro_f1"], name=row0["step"])
BASELINE = dict(row=row0, pr=pr_best)
# Each step may try more than one variant. Order matters: the label cleaning is tried on the pixel-only inputs first, because once
# neighbourhood inputs are used, a model that never saw field-edge pixels can be confused by the mixed surroundings of edge pixels.
STEPS = [("1  + 4 indices + observed-month flag", lambda c: [(c["ch"] + CH_IDX, c["clean"], "")]),
         ("2  + clean training labels",           lambda c: [(c["ch"], True, "")] if not c["clean"] else []),
         ("3  + neighbourhood inputs",            lambda c: [(c["ch"] + CH_NBR, c["clean"], "")] + ([(c["ch"] + CH_NBR, False, " (labels not cleaned)")] if c["clean"] else []))]
for label, variants in STEPS:
    best_try = None
    for ch, clean, note in variants(cur):
        row, S, est, pr = run_step(label + note, ch, clean); row["decision"] = "dropped (validation did not improve enough)"
        LADDER.append(row)
        if best_try is None or row["val_macro_f1"] > best_try["row"]["val_macro_f1"]:
            best_try = dict(row=row, ch=ch, clean=clean, S=S, est=est, pr=pr)
        else:
            del S, est
        gc.collect()
    if best_try is not None and best_try["row"]["val_macro_f1"] >= cur["val"] + CFG["MIN_GAIN"]:
        best_try["row"]["decision"] = "KEPT"; cur = dict(ch=best_try["ch"], clean=best_try["clean"], val=best_try["row"]["val_macro_f1"], name=best_try["row"]["step"])
        S_best, est_best, pr_best = best_try["S"], best_try["est"], best_try["pr"]
    elif best_try is not None:
        del best_try
    gc.collect()
LAD = pd.DataFrame(LADDER); display(LAD); LAD.to_csv(OUT / "tables" / "ladder.csv", index=False)
print("chosen configuration:", cur["name"], "| inputs per month:", len(cur["ch"]), "| clean training labels:", cur["clean"])
PROBS["LightGBM"] = pr_best

[  3.0 min] 0  benchmark setup (raw inputs): val F1 0.7308 | test acc 0.9034 | test macro-F1 0.7593
[  3.9 min] 1  + 4 indices + observed-month flag: val F1 0.6951 | test acc 0.8886 | test macro-F1 0.7183
[  4.4 min] 2  + clean training labels: val F1 0.6812 | test acc 0.9089 | test macro-F1 0.7222
[  8.1 min] 3  + neighbourhood inputs: val F1 0.7424 | test acc 0.9122 | test macro-F1 0.7798


,step,inputs_per_month,clean_train_labels,rounds,val_macro_f1,test_acc,test_macro_f1,minutes,decision
0,0 benchmark setup (raw inputs),11,False,285,0.7308,0.9034,0.7593,2.3527,start
1,1 + 4 indices + observed-month flag,16,False,66,0.6951,0.8886,0.7183,0.9597,dropped (validation did not improve enough)
2,2 + clean training labels,11,True,36,0.6812,0.9089,0.7222,0.4194,dropped (validation did not improve enough)
3,3 + neighbourhood inputs,19,False,302,0.7424,0.9122,0.7798,3.7118,KEPT


chosen configuration: 3  + neighbourhood inputs | inputs per month: 19 | clean training labels: False


## 7. Other models on the chosen inputs, then average them

In [18]:
S = S_best
try:
    t = time.time(); rf = RandomForestClassifier(n_estimators=CFG["RF_TREES"], max_depth=14, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=CFG["SEED"])
    rf.fit(flat(S["Xtr"]), S["ytr"]); PROBS["Random Forest"] = probs_of(rf, S); del rf
    log(f"Random Forest done in {(time.time() - t) / 60:.1f} min")
except Exception as e:
    print("Random Forest skipped:", repr(e))
try:
    t = time.time(); mlp = fit_mlp(S, CFG["SEED"])
    PROBS["MLP"] = dict(val=mlp_probs(mlp, S["Xva"]), test=mlp_probs(mlp, S["Xte"])); del mlp
    log(f"MLP done in {(time.time() - t) / 60:.1f} min")
except Exception as e:
    print("MLP skipped:", repr(e))
if len(PROBS) > 1:
    PROBS["Average of all models"] = {k: np.mean([v[k] for n, v in PROBS.items() if not n.startswith("Average")], axis=0) for k in ("val", "test")}
rows = []
for n, v in PROBS.items():
    mv, mt = metrics(S["yva"], v["val"].argmax(1)), metrics(S["yte"], v["test"].argmax(1))
    rows.append(dict(model=n, val_macro_f1=mv["f1"], test_acc=mt["acc"], test_macro_f1=mt["f1"], test_weighted_f1=mt["wf1"]))
MOD = pd.DataFrame(rows); display(MOD); MOD.to_csv(OUT / "tables" / "models.csv", index=False)
FINAL_NAME = MOD.sort_values("val_macro_f1", ascending=False).iloc[0]["model"]          # chosen on validation only
print("final model (best validation macro-F1):", FINAL_NAME)

[ 10.0 min] Random Forest done in 1.9 min
[ 10.2 min] MLP done in 0.2 min


,model,val_macro_f1,test_acc,test_macro_f1,test_weighted_f1
0,LightGBM,0.7424,0.9122,0.7798,0.9161
1,Random Forest,0.6895,0.8895,0.7209,0.9002
2,MLP,0.7104,0.8995,0.7382,0.9080
3,Average of all models,0.7354,0.9111,0.7689,0.9166


final model (best validation macro-F1): LightGBM


## 8. Neighbour smoothing (post-processing, reported separately)
Each pixel takes the average class probabilities of the pixels around it. The window size is chosen on the **validation** blocks.

In [19]:
def smooth_pred(probs, split_name, win):
    if win == 1: return probs.argmax(1)
    rows, cols = ROWS[IDX[split_name]], COLS[IDX[split_name]]; out = np.empty_like(probs)
    for k in range(K):
        m = np.zeros((H, W), np.float32); m[rows, cols] = probs[:, k]
        out[:, k] = uniform_filter(m, size=win, mode="constant")[rows, cols]
    return out.argmax(1)

fin = PROBS[FINAL_NAME]; sm_rows = []
for w in CFG["SMOOTH_WINDOWS"]:
    sm_rows.append(dict(window=w, val_macro_f1=metrics(S["yva"], smooth_pred(fin["val"], "val", w))["f1"],
                        test_acc=metrics(S["yte"], smooth_pred(fin["test"], "test", w))["acc"], test_macro_f1=metrics(S["yte"], smooth_pred(fin["test"], "test", w))["f1"]))
SM = pd.DataFrame(sm_rows); display(SM); SM.to_csv(OUT / "tables" / "smoothing.csv", index=False)
BEST_WIN = int(SM.sort_values(["val_macro_f1", "window"], ascending=[False, True]).iloc[0]["window"])
print("smoothing window chosen on validation:", BEST_WIN)

,window,val_macro_f1,test_acc,test_macro_f1
0,1,0.7424,0.9122,0.7798
1,3,0.7560,0.9217,0.8034
2,5,0.7501,0.9195,0.8002
3,7,0.7407,0.9158,0.7961
4,9,0.7332,0.9101,0.7875
5,11,0.7261,0.9038,0.7801


smoothing window chosen on validation: 3


## 9. RESULTS

In [20]:
y_te = S["yte"]; te_idx = IDX["test"]
p_base = BASELINE["pr"]["test"].argmax(1); p_fin = fin["test"].argmax(1); p_sm = smooth_pred(fin["test"], "test", BEST_WIN)
is_edge = EDGE_AT[te_idx]
GROUP = {"Alfalfa": "Hay", "Other Hay": "Hay", "Corn": "Corn (field+sweet)", "Sweet Corn": "Corn (field+sweet)",
         "Winter Wheat": "Wheat (winter+spring)", "Spring Wheat": "Wheat (winter+spring)", "Durum Wheat": "Wheat (winter+spring)"}
gnames = sorted({GROUP.get(n, n) for n in CLASS_NAMES}); gmap = np.array([gnames.index(GROUP.get(n, n)) for n in CLASS_NAMES])
def grouped(y, p):
    pr, rc, f1, s = precision_recall_fscore_support(gmap[y], gmap[p], labels=np.arange(len(gnames)), zero_division=0)
    return dict(acc=accuracy_score(gmap[y], gmap[p]), f1=float(f1[s > 0].mean()))

lines = []
def say(s=""): print(s); lines.append(s)
say("===== RESULTS =====")
say(f"area {CFG['AOI']} | classes {K} | test pixels {len(te_idx):,} | settings: GBM_ROUNDS={CFG['GBM_ROUNDS']} RF_TREES={CFG['RF_TREES']} MAX_TRAIN_PER_CLASS={CFG['MAX_TRAIN_PER_CLASS']}")
say(f"chosen on validation: inputs per month {len(cur['ch'])} (raw {C}), clean training labels = {cur['clean']}, final model = {FINAL_NAME}, smoothing window = {BEST_WIN}")
say()
say("A. Ladder (LightGBM, test blocks, all pixels):"); say(LAD[["step", "inputs_per_month", "val_macro_f1", "test_acc", "test_macro_f1", "decision"]].to_string(index=False))
say(); say("B. Models on the chosen inputs:"); say(MOD.to_string(index=False))
say(); say(f"C. Smoothing window (chosen on validation = {BEST_WIN}):"); say(SM.to_string(index=False))
head = pd.DataFrame([
    dict(system="benchmark setup (LightGBM, raw inputs)", **{k: v for k, v in metrics(y_te, p_base).items() if k in ("acc", "f1")}),
    dict(system=f"improved: {FINAL_NAME}", **{k: v for k, v in metrics(y_te, p_fin).items() if k in ("acc", "f1")}),
    dict(system=f"improved + neighbour smoothing (window {BEST_WIN})", **{k: v for k, v in metrics(y_te, p_sm).items() if k in ("acc", "f1")})]).rename(columns={"acc": "test_accuracy", "f1": "test_macro_f1"})
say(); say(f"D. HEADLINE (all test pixels, {K}-class scheme):"); say(head.to_string(index=False)); head.to_csv(OUT / "tables" / "headline.csv", index=False)
say(); say("E. Where the remaining errors are (improved + smoothing):")
ed = pd.DataFrame([dict(pixels="interior (not on a field edge)", share_pct=100 * (~is_edge).mean(), accuracy=accuracy_score(y_te[~is_edge], p_sm[~is_edge])),
                   dict(pixels="field edge (mixed pixels)", share_pct=100 * is_edge.mean(), accuracy=accuracy_score(y_te[is_edge], p_sm[is_edge]) if is_edge.any() else np.nan)])
say(ed.to_string(index=False))
g0, g1, g2 = grouped(y_te, p_base), grouped(y_te, p_fin), grouped(y_te, p_sm)
say(); say(f"F. Same predictions with similar crops grouped after prediction ({len(gnames)} groups: Hay = Alfalfa+Other Hay, Corn = field+sweet, Wheat = winter+spring). This is a different, easier task and is NOT comparable with D:")
say(f"   benchmark setup: acc {g0['acc']:.4f} macro-F1 {g0['f1']:.4f} | improved: acc {g1['acc']:.4f} macro-F1 {g1['f1']:.4f} | improved + smoothing: acc {g2['acc']:.4f} macro-F1 {g2['f1']:.4f}")
pc = per_class(y_te, p_base).set_index("cls")[["f1"]].rename(columns={"f1": "f1_benchmark_setup"})
pc["f1_improved_smoothed"] = per_class(y_te, p_sm).set_index("cls")["f1"]; pc["recall_improved_smoothed"] = per_class(y_te, p_sm).set_index("cls")["recall"]
pc["test_pixels"] = per_class(y_te, p_sm).set_index("cls")["support"]
say(); say("G. Per class (test):"); say(pc.round(3).to_string()); pc.to_csv(OUT / "tables" / "per_class.csv")
cm = pd.DataFrame(confusion_matrix(y_te, p_sm, labels=np.arange(K)), index=CLASS_NAMES, columns=CLASS_NAMES)
say(); say("H. Confusion matrix of improved + smoothing (rows = CDL label, columns = predicted):"); say(cm.to_string()); cm.to_csv(OUT / "tables" / "confusion_final.csv")
try:
    gain = pd.Series(est_best.booster_.feature_importance("gain"), index=[f"{MONTHS[t][0]}-{MONTHS[t][1]:02d}_{ALL_CH[c]}" for t in range(T) for c in S["ch"]]).sort_values(ascending=False)
    say(); say("I. Ten most important inputs (LightGBM gain):"); say((gain / gain.sum()).head(10).round(4).to_string())
except Exception as e:
    print("importance skipped:", repr(e))
np.savez_compressed(OUT / "predictions" / "final_test.npz", rows=ROWS[te_idx], cols=COLS[te_idx], y_true=y_te, y_benchmark=p_base, y_improved=p_fin, y_smoothed=p_sm)
(OUT / "RESULTS.txt").write_text("\n".join(lines))
say(); say(f"finished in {(time.time() - T0) / 60:.1f} minutes. Copy everything from '===== RESULTS =====' down to here and send it.")

===== RESULTS =====
area 2 | classes 9 | test pixels 144,922 | settings: GBM_ROUNDS=400 RF_TREES=100 MAX_TRAIN_PER_CLASS=30000
chosen on validation: inputs per month 19 (raw 11), clean training labels = False, final model = LightGBM, smoothing window = 3

A. Ladder (LightGBM, test blocks, all pixels):
                                step  inputs_per_month  val_macro_f1  test_acc  test_macro_f1                                    decision
     0  benchmark setup (raw inputs)                11        0.7308    0.9034         0.7593                                       start
1  + 4 indices + observed-month flag                16        0.6951    0.8886         0.7183 dropped (validation did not improve enough)
          2  + clean training labels                11        0.6812    0.9089         0.7222 dropped (validation did not improve enough)
           3  + neighbourhood inputs                19        0.7424    0.9122         0.7798                                        KEPT

B. Mod